In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Enjoy synthetic 2D fields

## Summary

Experiment employing the synthetic fields. This includes: Plotting, adding CRS information, and reprojections (together w/ their plots).

## Results

### Key results

- Helpful functionality:
  - Equip synthetic fields w/ CRS using an additional function `phoibe.synthetic_data.fields.make_field_rio`.

### Details


## Remarks

- Reprojections awaits finetuning regarding logging. You may want to readjust the logging level to `INFO` instead of `DEBUG`.

In [ ]:
import logging

import matplotlib
import numpy as np
import xarray

import phoibe

LOGGER = logging.getLogger("phoibe.geography.crs.reproject")
LOGGER.setLevel(level=logging.DEBUG)


LAND_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list(
    "land_cmap", [(0.00, "#2e7d32"), (0.30, "#66bb6a"), (0.55, "#a1887f"), (0.75, "#9e9e9e"), (1.00, "#ffffff")]
)

## Create examplary field

1. Create vanilla eggbox field.
2. Add geographic coordinates at some region.
3. Show the respective plots.

In [ ]:
WIDTH, HEIGHT = 30, 20
eggbox = phoibe.synthetic_data.fields.make_eggbox_field(nx=WIDTH, ny=HEIGHT, dx=0.1, dy=0.1, freq_x=3, freq_y=3)
figure, ax = phoibe.geography.plot.plot_raster(da=eggbox)

bounds = 8, 48, 9, 49
eggbox = phoibe.synthetic_data.fields.make_field_rio(da=eggbox, bounds=bounds, crs=4326, dtype="float", nodata=np.nan)

phoibe.geography.plot.plot_raster(da=eggbox, title=f"CRS: {eggbox.rio.crs.to_authority()}")

## Reproject

1. Reproject to UTM.
2. Reproject back to GCS.

Both including changing the resolution.

In [ ]:
eggbox_utm, summary = phoibe.geography.crs.reproject_rasterdata(da=eggbox, crs_to=32632, resolution=30)
phoibe.geography.plot.plot_raster(da=eggbox_utm)

eggbox_utm_gcs, summary = phoibe.geography.crs.reproject_rasterdata(da=eggbox_utm, crs_to=4326, resolution=0.1)
phoibe.geography.plot.plot_raster(da=eggbox_utm_gcs)

display(summary)

In [ ]:
def print_min_mean_max(da: xarray.DataArray, coord: str):
    values = getattr(da, coord)
    print(float(values.min()), float(values.mean()), float(values.max()), len(values))


print_min_mean_max(eggbox, "x")
print_min_mean_max(eggbox, "y")

print_min_mean_max(eggbox_utm, "x")
print_min_mean_max(eggbox_utm, "y")

print_min_mean_max(eggbox_utm_gcs, "x")
print_min_mean_max(eggbox_utm_gcs, "y")